# Goodreads content-based recommender with TF-IDF

This notebook builds a **content-based book recommender** from Goodreads metadata.
The model represents each book as a TF-IDF vector built from textual attributes such as title, author, genre, and description.

The notebook is organized into five parts:

1. **Load and inspect the source data**
2. **Compare the available datasets**
3. **Build a clean text representation for each book**
4. **Fit a reusable TF-IDF recommender**
5. **Query the model by `book_id` or by title**

The recommender is item-based. It recommends books whose metadata are most similar to the seed book or seed books.
Similarity is computed only for the query, so the notebook avoids creating a full item-by-item similarity matrix.

In [1]:
import pandas as pd
import numpy as np
from typing import Optional, Dict, Iterable, Set, List

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import linear_kernel

In [2]:
books_kaggle = pd.read_csv('../data/raw/Books.csv', low_memory=False)
books_kaggle.head()

,ISBN,Book-Title,Book-Author,Year-Of-Publication,Publisher,Image-URL-S,Image-URL-M,Image-URL-L
0,0195153448,Classical Mythology,Mark P. O. Morford,2002,Oxford University Press,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...,http://images.amazon.com/images/P/0195153448.0...
1,0002005018,Clara Callan,Richard Bruce Wright,2001,HarperFlamingo Canada,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...,http://images.amazon.com/images/P/0002005018.0...
2,0060973129,Decision in Normandy,Carlo D'Este,1991,HarperPerennial,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...,http://images.amazon.com/images/P/0060973129.0...
3,0374157065,Flu: The Story of the Great Influenza Pandemic...,Gina Bari Kolata,1999,Farrar Straus Giroux,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...,http://images.amazon.com/images/P/0374157065.0...
4,0393045218,The Mummies of Urumchi,E. J. W. Barber,1999,W. W. Norton &amp; Company,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...,http://images.amazon.com/images/P/0393045218.0...


In [3]:
# we will use our already cleaned kaggle dataset with some added columns
books_kaggle = pd.read_csv('../data/cleaned/cleaned_books.csv')

In [4]:
books_goodreads = pd.read_csv("../data/goodreads_data/dataset_goodreads_filtered_description.csv")
books_goodreads.head()

,title,book_id,average_rating,similar_books,description,author_id,isbn13,url,image_url,ratings_count,genre,author_name
0,"The Hunger Games (The Hunger Games, #1)",2767052,4.34,"c(""""1902241"""", """"146499"""", """"954674"""", """"99179...",Winning will make you famous.\nLosing means ce...,153394,9780439023481,https://www.goodreads.com/book/show/2767052-th...,https://images.gr-assets.com/books/1447303603m...,4899965,YA,Suzanne Collins
1,Harry Potter and the Sorcerer's Stone (Harry P...,3,4.45,"c(""""13830"""", """"127586"""", """"121822"""", """"37586""""...",Harry Potter's life is miserable. His parents ...,1077326,9780439554930,https://www.goodreads.com/book/show/3.Harry_Po...,https://images.gr-assets.com/books/1474154022m...,4765497,fantasy,J.K. Rowling
2,"Twilight (Twilight, #1)",41865,3.57,"c(""""1326258"""", """"140077"""", """"35729"""", """"301234...",About three things I was absolutely positive.\...,941441,9780316015844,https://www.goodreads.com/book/show/41865.Twil...,https://images.gr-assets.com/books/1361039443m...,3941381,"fantasy,YA",Stephenie Meyer
3,To Kill a Mockingbird,2657,4.26,"c(""""1934"""", """"2156"""", """"15638"""", """"53835"""", """"...",The unforgettable novel of a childhood in a sl...,1825,9780061120084,https://www.goodreads.com/book/show/2657.To_Ki...,https://images.gr-assets.com/books/1361975680m...,3255518,history_biography,Harper Lee
4,The Fault in Our Stars,11870085,4.26,"c(""""10051706"""", """"11418182"""", """"10327303"""", """"...","There is an alternate cover edition .\n""""I fe...",1406384,9780525478812,https://www.goodreads.com/book/show/11870085-t...,https://images.gr-assets.com/books/1360206420m...,2429317,YA,John Green


In [5]:
books_goodreads.columns

Index(['title', 'book_id', 'average_rating', 'similar_books', 'description',
       'author_id', 'isbn13', 'url', 'image_url', 'ratings_count', 'genre',
       'author_name'],
      dtype='object')

### Selecting the working dataset

The Goodreads dataset contains richer text features than the Kaggle dataset, especially descriptions and genre information.
Because the goal of this notebook is content-based recommendation, Goodreads is the more suitable source for the final model.

### Compare the available datasets

Before building the recommender, we briefly compare the cleaned Kaggle catalog and the Goodreads catalog.
This helps confirm which dataset provides the textual features needed for a TF-IDF model.

In [6]:
print("Books from Kaggle dataset: ", books_kaggle.shape[0])
print("Books from Goodreads dataset: ", books_goodreads.shape[0])

Books from Kaggle dataset:  271307
Books from Goodreads dataset:  15591


The Goodreads dataset has already been filtered to keep books with sufficient rating activity.
As a result, it contains fewer rows than the Kaggle catalog, but each row is typically more useful for recommendation because the textual metadata are richer and the items are better supported.

In [7]:
# check the number of common books in both datasets by comparing the ISBN numbers
common_books = set(books_kaggle['isbn']).intersection(set(books_goodreads['isbn13']))
common_books

set()

The two datasets are not directly aligned by ISBN because the Kaggle catalog mainly stores **ISBN-10** values while the Goodreads data use **ISBN-13**.
A strict ISBN match would therefore miss many valid overlaps.

For a quick comparison, this notebook also checks overlap by normalized title.
This is not used for the final recommender, but it is useful for understanding how much the catalogs have in common.

In [8]:
from itertools import islice

def clean_title(title):
    return title.strip().lower()

books_kaggle_titles = set(books_kaggle['title'].apply(clean_title))
books_goodreads_titles = set(books_goodreads['title'].apply(clean_title))
common_books_titles = books_kaggle_titles.intersection(books_goodreads_titles)
print("Number of common books based on titles: ", len(common_books_titles))
print("Examples of matching normalized titles:")
list(islice(common_books_titles, 10))

Number of common books based on titles:  2008
Examples of matching normalized titles:


['black beauty',
 'true betrayals',
 'airport',
 'the god project',
 'the scarlet pimpernel',
 'falling angels',
 'a dangerous fortune',
 'paperweight',
 'shanna',
 'the night stalker']

The final recommender in this notebook uses the Goodreads catalog directly.
That choice keeps the pipeline simple and makes full use of the fields that are most informative for content similarity, especially **description**, **genre**, **title**, and **author**.

In [9]:
books = books_goodreads.copy()
books.isnull().sum()

title                0
book_id              0
average_rating       0
similar_books        0
description          0
author_id            0
isbn13            1470
url                  0
image_url            0
ratings_count        0
genre                0
author_name          0
dtype: int64

A quick inspection shows that the Goodreads dataset already contains the core fields needed for a content-based model.
The next step is to clean those text fields and combine them into a single representation that can be vectorized with TF-IDF.

## TF-IDF content-based recommender

`TfidfBookRecommender` is a **content-based recommender** that represents each book by a TF-IDF vector built from textual metadata.

In this notebook, the model does not work directly with ratings or user-user interactions.
Instead, it uses a precomputed text representation of each book, stored in a column such as `content`, and recommends books whose text representation is similar to the seed books.

The overall idea is simple:

1. build one text document for each book
2. convert those documents into TF-IDF vectors
3. measure similarity between books with cosine similarity
4. return the most similar books to the seed books

---

### 1. Book representation

Before fitting the recommender, the notebook creates a text column with `build_book_content(...)`.

This combines selected metadata fields such as:

- title
- author
- genre
- description

into one text document per book.

Conceptually, the representation looks like:

$$
d_i = w_t \cdot \text{title}_i + w_a \cdot \text{author}_i + w_g \cdot \text{genre}_i + w_d \cdot \text{description}_i
$$

where the weights are implemented by repeating the text of each field before concatenation.

This means that fields such as **genre** or **description** can have stronger influence on similarity than shorter fields such as title.

---

### 2. TF-IDF vectorization

After the text column is prepared, the recommender fits a TF-IDF vectorizer.

For a term $t$ in document $d$, TF-IDF is:

$$
\text{tfidf}(t,d) = \text{tf}(t,d) \cdot \text{idf}(t)
$$

with inverse document frequency typically defined as:

$$
\text{idf}(t) = \log\left(\frac{N + 1}{\text{df}(t) + 1}\right) + 1
$$

where:

- $N$ is the number of documents
- $\text{df}(t)$ is the number of documents containing term $t$

This gives higher importance to words that are frequent in one book but not common across all books.

The result is a sparse matrix:

$$
X \in \mathbb{R}^{n \times p}
$$

where:

- $n$ is the number of books
- $p$ is the number of TF-IDF features

Because the vectorizer uses L2 normalization, each row vector has unit length.

---

### 3. Book similarity

For a seed book $i$ and candidate book $j$, similarity is computed by cosine similarity:

$$
\text{sim}(i,j) =
\frac{x_i^\top x_j}{\|x_i\| \, \|x_j\|}
$$

Since the TF-IDF vectors are already L2-normalized, this reduces to a simple dot product:

$$
\text{sim}(i,j) = x_i^\top x_j
$$

Books with more similar words, phrases, genres, authors, or descriptions will therefore have higher similarity.

---

### 4. Multiple seed books

The recommender supports one or more seed books.

For each seed book, it computes a similarity vector to all books.
These similarity vectors are then aggregated.

Two aggregation modes are supported:

#### Mean aggregation

$$
s_j = \frac{1}{M} \sum_{m=1}^{M} \text{sim}(i_m, j)
$$

This favors books that are reasonably similar to **all** seed books.

#### Max aggregation

$$
s_j = \max_{m=1,\dots,M} \text{sim}(i_m, j)
$$

This favors books that are very similar to **at least one** seed book.

Here:

- $M$ is the number of seed books
- $i_m$ is the $m$-th seed book
- $s_j$ is the final similarity score of candidate book $j$

In the implementation, `"max"` is the default.

---

### 5. Excluding the seed books

If `exclude_seeds=True`, the seed books themselves are removed from the recommendation list by assigning them a very low similarity score before ranking.

This prevents the recommender from returning the input books as recommendations.

---

### 6. Optional reranking

The class also supports an optional lightweight reranking step.

After retrieving the top TF-IDF matches, it can combine similarity with metadata quality signals:

- average rating
- popularity, measured by rating count

These are normalized into the interval $[0,1]$ and combined with similarity as:

$$
\text{score}_j =
\text{similarity}_j + 0.10 \cdot r_j^{\text{norm}} + 0.05 \cdot p_j^{\text{norm}}
$$

where:

- $r_j^{\text{norm}}$ is normalized average rating
- $p_j^{\text{norm}}$ is normalized popularity

So reranking slightly promotes books that are not only similar in content, but also better rated or more popular.

---

This method is especially useful when:

- we want recommendations based on **book content**
- we have good metadata such as description and genre
- user interaction data are sparse or unavailable
- we want to recommend books even for a completely new user

Its main limitation is that it can only recommend based on what is present in the text representation.
So it cannot learn hidden preference patterns from large-scale user behavior the way collaborative methods can.

---

### Practical meaning of the main parameters

- `min_df` removes very rare words
- `max_df` removes overly common words
- `ngram_range=(1,2)` uses both unigrams and bigrams
- `max_features` limits vocabulary size
- `agg="max"` favors similarity to any one seed
- `agg="mean"` favors similarity across all seeds
- `rerank=True` slightly boosts highly rated and popular similar books

---

This method recommends books whose textual metadata representation is most similar to the seed books in TF-IDF space.

In [10]:
def _clean_text_series(s: pd.Series) -> pd.Series:
    """
    Standardize a text column before it is used in the content model.

    The function performs four simple preprocessing steps:
    1. replace missing values with empty strings
    2. convert all values to strings
    3. lowercase the text
    4. collapse repeated whitespace and trim the result
    """
    s = s.fillna("").astype(str).str.lower()
    s = s.str.replace(r"\s+", " ", regex=True).str.strip()
    return s

def build_book_content(
    df: pd.DataFrame,
    *,
    field_weights: Optional[Dict[str, int]] = None,
) -> pd.Series:
    """
    Combine several book attributes into one text field for TF-IDF vectorization.

    The function expects the Goodreads-style columns:
    `title`, `description`, `author_name`, and `genre`.

    Each field can be given a weight. Weights are implemented by repeating the
    text of that field multiple times before concatenation. This simple approach
    increases the influence of selected attributes in the final TF-IDF vector.
    """
    if field_weights is None:
        field_weights = {
            "title": 0,
            "author": 1,
            "genre": 3,
            "description": 3,
        }

    required = ["title", "description", "author_name", "genre"]
    missing = [c for c in required if c not in df.columns]
    if missing:
        raise ValueError(f"Missing required columns: {missing}")

    title  = _clean_text_series(df["title"])
    author = _clean_text_series(df["author_name"])
    genre  = _clean_text_series(df["genre"])
    desc   = _clean_text_series(df["description"])

    wt = max(1, int(field_weights.get("title", 1)))
    wa = max(1, int(field_weights.get("author", 1)))
    wg = max(1, int(field_weights.get("genre", 1)))
    wd = max(1, int(field_weights.get("description", 1)))

    out = (title + " ") * wt + (author + " ") * wa + (genre + " ") * wg + (desc + " ") * wd
    return out.str.replace(r"\s+", " ", regex=True).str.strip()

In [11]:
class TfidfBookRecommender:
    """
    Content-based recommender built on top of TF-IDF vectors.

    The recommender expects a precomputed text column that represents each book
    as a single document, for example a concatenation of title, author, genre,
    and description. It fits a TF-IDF vectorizer on that text and recommends
    books by cosine similarity in TF-IDF space.

    The class exposes two public query methods:
    - `recommend_by_ids(...)`
    - `recommend_by_title(...)`

    Parameters
    ----------
    id_col : str, default="book_id"
    title_col : str, default="title"
    rating_col : str, default="average_rating"
    popularity_col : str, default="ratings_count"

    stop_words : str, list, or None, default="english"
        Stop-word setting passed to `TfidfVectorizer`.

    min_df : int or float, default=2
        Minimum document frequency for terms retained by the vectorizer.

    max_df : int or float, default=0.9
        Maximum document frequency for terms retained by the vectorizer.

    ngram_range : tuple[int, int], default=(1, 2)
        Lower and upper boundary of n-grams used by the vectorizer.

    max_features : int, default=200_000
        Maximum number of TF-IDF features retained.
    """
    def __init__(
        self,
        *,
        id_col="book_id",
        title_col="title",
        rating_col="average_rating",
        popularity_col="ratings_count",
        stop_words="english",
        min_df=2,
        max_df=0.9,
        ngram_range=(1, 2),
        max_features=200_000,
    ):
        self.id_col = id_col
        self.title_col = title_col
        self.rating_col = rating_col
        self.popularity_col = popularity_col

        self.vectorizer = TfidfVectorizer(
            stop_words=stop_words,
            min_df=min_df,
            max_df=max_df,
            ngram_range=ngram_range,
            max_features=max_features,
            strip_accents="unicode",
            lowercase=True,
            sublinear_tf=True,
            norm="l2",
        )

        self.df_ = None
        self.X_ = None
        self.id_to_idx_ = None
        self.title_to_indices_ = None

    def fit(self, df: pd.DataFrame, *, content_col: str):
        """
        Fit the TF-IDF recommender on a prepared content column.

        The method:
        1. validates the ID column
        2. standardizes the relevant columns
        3. removes rows with empty content
        4. fits the TF-IDF vectorizer
        5. stores lookup tables for IDs and titles

        Parameters
        ----------
        df : pd.DataFrame
            Input catalog containing book metadata and a prepared text column.

        content_col : str
            Name of the column containing one text document per book.

        Returns
        -------
        TfidfBookRecommender
            The fitted recommender instance.
        """
        if self.id_col not in df.columns:
            raise ValueError(f"Missing id_col={self.id_col!r} in df")

        data = df.copy()

        data[self.id_col] = data[self.id_col].astype(str)
        if self.title_col in data.columns:
            data[self.title_col] = data[self.title_col].fillna("").astype(str)

        data[content_col] = data[content_col].fillna("").astype(str)

        keep = data[content_col].str.len() > 0
        data = data.loc[keep].reset_index(drop=True)

        X = self.vectorizer.fit_transform(data[content_col])

        self.df_ = data
        self.X_ = X
        self.id_to_idx_ = pd.Series(data.index.values, index=data[self.id_col]).to_dict()

        if self.title_col in data.columns:
            key = data[self.title_col].str.lower()
            self.title_to_indices_ = key.groupby(key).apply(lambda s: s.index.to_list()).to_dict()
        else:
            self.title_to_indices_ = {}

        return self

    def _resolve_title_to_idx(self, title: str) -> int:
        """
        Resolve a title query to a single row index in the fitted catalog.

        If multiple rows share the same title, the method chooses the most
        supported match using metadata priority:
        1. highest popularity
        2. otherwise highest average rating
        3. otherwise first matching row

        Parameters
        ----------
        title : str
            Title to resolve.

        Returns
        -------
        int
            Row index of the selected catalog entry.
        """
        if not self.title_to_indices_:
            raise ValueError("No title column available to search by title.")

        key = (title or "").strip().lower()
        if key not in self.title_to_indices_:
            raise KeyError(f"Title not found: {title!r}")

        idxs = self.title_to_indices_[key]
        if len(idxs) == 1:
            return idxs[0]

        df = self.df_.loc[idxs]
        if self.popularity_col in df.columns:
            return int(df[self.popularity_col].fillna(0).astype(float).idxmax())
        if self.rating_col in df.columns:
            return int(df[self.rating_col].fillna(0).astype(float).idxmax())
        return int(idxs[0])

    def recommend_by_ids(
        self,
        seed_book_ids,
        *,
        k=10,
        agg="max",  # "max" or "mean"
        rerank=False,
        exclude_seeds=True,
    ) -> pd.DataFrame:
        """
        Recommend books similar to one or more seed book IDs.

        For each seed book, cosine similarity is computed against all books in
        TF-IDF space. The resulting similarity vectors are aggregated either by
        maximum or mean.

        Parameters
        ----------
        seed_book_ids : str, int, or sequence
            One seed book ID or multiple seed book IDs.

        k : int, default=10
            Number of recommendations to return.

        agg : {"max", "mean"}, default="max"
            Aggregation mode across multiple seed books.
            - "max": keep the highest similarity from any seed
            - "mean": average similarity across seeds

        rerank : bool, default=False
            If True, rerank the top TF-IDF matches using a small bonus from
            average rating and popularity metadata.

        exclude_seeds : bool, default=True
            If True, remove the seed books themselves from the result list.

        Returns
        -------
        pd.DataFrame
            Ranked recommendation table with similarity and optionally metadata.
        """
        if isinstance(seed_book_ids, (str, int)):
            seed_book_ids = [seed_book_ids]
        seed_ids = [str(x) for x in seed_book_ids]

        seed_idxs = []
        for bid in seed_ids:
            if bid in self.id_to_idx_:
                seed_idxs.append(self.id_to_idx_[bid])
        if not seed_idxs:
            raise KeyError("None of the provided seed_book_ids exist in the model index.")

        sims_list = []
        for idx in seed_idxs:
            sims = linear_kernel(self.X_[idx], self.X_).ravel()
            sims_list.append(sims)

        sims_stack = np.vstack(sims_list)
        if agg == "mean":
            sims_final = sims_stack.mean(axis=0)
        else:
            sims_final = sims_stack.max(axis=0)

        if exclude_seeds:
            sims_final[seed_idxs] = -1.0

        k = int(k)
        k = min(k, len(sims_final))
        top = np.argpartition(-sims_final, kth=k-1)[:k]
        top = top[np.argsort(-sims_final[top])]

        out = self.df_.iloc[top].copy()
        out["similarity"] = sims_final[top]

        if rerank:
            r = out[self.rating_col].fillna(0).astype(float) if self.rating_col in out.columns else 0.0
            p = out[self.popularity_col].fillna(0).astype(float) if self.popularity_col in out.columns else 0.0

            r = (r - r.min()) / (r.max() - r.min() + 1e-9)
            p = (p - p.min()) / (p.max() - p.min() + 1e-9)
            out["score"] = out["similarity"] + 0.10 * r + 0.05 * p
            out = out.sort_values(["score"], ascending=False)

        cols = [c for c in [self.id_col, self.title_col, "similarity", self.rating_col, self.popularity_col] if c in out.columns]
        return out[cols].reset_index(drop=True)

    def recommend_by_title(self, titles, *, k=10, **kwargs) -> pd.DataFrame:
        """
        Recommend books similar to a title already present in the catalog.
        """
        if isinstance(titles, str):
            titles = [titles]

        seed_ids = []

        for i, title in enumerate(titles):
            idx = self._resolve_title_to_idx(title)
            seed_id = self.df_.iloc[idx][self.id_col]
            seed_ids.append(seed_id)

        return self.recommend_by_ids(seed_ids, k=k, **kwargs)

## Build and fit the model

In this section, the notebook creates the final `content` field, initializes the recommender, and fits the TF-IDF vectorizer to the Goodreads catalog.

In [12]:
books = books.copy()
books["content"] = build_book_content(books)

rec = TfidfBookRecommender(
    id_col="book_id",
    title_col="title",
    rating_col="average_rating",
    popularity_col="ratings_count",
    min_df=2,
    max_df=0.9,
    ngram_range=(1, 2),
)

rec.fit(books, content_col="content")

## Query the recommender

The recommender provides two query methods:

- `recommend_by_ids(...)` for exact item lookup using `book_id`
- `recommend_by_title(...)` for convenience when you only know the title

Using `book_id` is generally safer because titles are not guaranteed to be unique.
When duplicate titles exist, the recommender resolves them to the most popular matching book.

### Interpreting the recommendation output

The returned table contains the most similar books found by the TF-IDF model.

- **`similarity`** is the cosine similarity between the query and each candidate.
- **`average_rating`** and **`ratings_count`** are metadata columns included for interpretation.
- When `rerank=True`, those metadata columns slightly adjust the final order after the similarity scores are computed.

For exploratory work, it is useful to compare results from `agg="max"` and `agg="mean"` on the same seed set because they emphasize different notions of similarity.

In [13]:
title = books.loc[books["book_id"] == 3, "title"].iloc[0]
title

"Harry Potter and the Sorcerer's Stone (Harry Potter, #1)"

In [14]:
# By book_id
# 3: Harry Potter and the Sorcerer's Stone (Harry Potter, #1)
rec.recommend_by_ids(["3"], k=10, agg="max", rerank=True)

,book_id,title,similarity,average_rating,ratings_count
0,5,Harry Potter and the Prisoner of Azkaban (Harr...,0.221430,4.53,1876252
1,2,Harry Potter and the Order of the Phoenix (Har...,0.203494,4.47,1766895
2,10,"Harry Potter Collection (Harry Potter, #1-6)",0.180117,4.73,25245
3,72193,Harry Potter and the Philosopher's Stone (Harr...,0.222853,4.45,31614
4,136251,Harry Potter and the Deathly Hallows (Harry Po...,0.141504,4.62,1784684
5,6,Harry Potter and the Goblet of Fire (Harry Pot...,0.145468,4.53,1792561
6,15881,Harry Potter and the Chamber of Secrets (Harry...,0.174080,4.38,1821802
7,28132722,Harry Potter and the Sorcerer's Stone (Harry P...,0.166427,4.45,19106
8,2002,Harry Potter Schoolbooks Box Set: Two Classic ...,0.154561,4.40,10914
9,31538647,Hogwarts: An Incomplete and Unreliable Guide (...,0.151433,4.22,15928


In [15]:
# By title
rec.recommend_by_title("The Hobbit", k=10, rerank=True)

,book_id,title,similarity,average_rating,ratings_count
0,30,J.R.R. Tolkien 4-Book Boxed Set: The Hobbit an...,0.257402,4.59,92172
1,659469,The Hobbit: Graphic Novel,0.229927,4.48,155239
2,34,The Fellowship of the Ring (The Lord of the Ri...,0.149384,4.34,1813229
3,33,"The Lord of the Rings (The Lord of the Rings, ...",0.132556,4.48,396933
4,67939,Bilbo's Last Song,0.167516,4.09,1819
5,7336,J.R.R. Tolkien: A Biography,0.133858,4.03,7819
6,92003,The Atlas of Middle-Earth,0.108882,4.15,10020
7,23617,Roverandom,0.134490,3.86,7180
8,597790,The Children of Húrin,0.097824,3.94,40570
9,1081560,"The History of the Hobbit, Part One: Mr. Baggins",0.084801,3.81,112069


In [16]:
# Multi-seed query
# 3: Harry Potter and the Sorcerer's Stone (Harry Potter, #1)
# 2767052: The Hunger Games (The Hunger Games, #1)
rec.recommend_by_ids(["3", "2767052"], k=15, agg="max", rerank=True)

,book_id,title,similarity,average_rating,ratings_count
0,5,Harry Potter and the Prisoner of Azkaban (Harr...,0.221430,4.53,1876252
1,2,Harry Potter and the Order of the Phoenix (Har...,0.203494,4.47,1766895
2,6148028,"Catching Fire (The Hunger Games, #2)",0.206953,4.30,1854746
3,12187797,Catching Fire (The Hunger Games #2),0.245195,4.30,4486
4,72193,Harry Potter and the Philosopher's Stone (Harr...,0.222853,4.45,31614
5,10,"Harry Potter Collection (Harry Potter, #1-6)",0.180117,4.73,25245
6,8574414,"Catching Fire (Hunger Games, #2)",0.236155,4.30,12717
7,15881,Harry Potter and the Chamber of Secrets (Harry...,0.174080,4.38,1821802
8,11873368,The World of the Hunger Games,0.175362,4.48,15554
9,11301989,The Hunger Games Trilogy,0.161381,4.49,23515
